In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# get iniago martinez's percentiles

## load 24-25 season data

In [13]:
df_raw = pd.read_csv('players_data-2024_2025.csv')
print(df_raw)

        Rk             Player   Nation    Pos        Squad  \
0        1         Max Aarons  eng ENG     DF  Bournemouth   
1        2         Max Aarons  eng ENG  DF,MF     Valencia   
2        3     Rodrigo Abajas   es ESP     DF     Valencia   
3        4     James Abankwah   ie IRL  DF,MF      Udinese   
4        5  Keyliane Abdallah   fr FRA     FW    Marseille   
...    ...                ...      ...    ...          ...   
2849  2850       Edhy Zuliani   fr FRA     DF     Toulouse   
2850  2851   Szymon Żurkowski   pl POL     MF       Empoli   
2851  2852    Martin Ødegaard   no NOR     MF      Arsenal   
2852  2853        Milan Đurić   ba BIH     FW        Monza   
2853  2854        Milan Đurić   ba BIH     FW        Parma   

                    Comp   Age    Born  MP  Starts  ...  Att (GK)  Thr  \
0     eng Premier League  24.0  2000.0   3       1  ...       NaN  NaN   
1             es La Liga  24.0  2000.0   4       1  ...       NaN  NaN   
2             es La Liga  21.0  2

## filtering

In [14]:
df_filtered = df_raw[df_raw['Min'] >= 900].copy()

df_cbs = df_filtered[df_filtered['Pos'].str.contains('DF', na=False, case=False)]

print(df_cbs)


        Rk            Player  Nation    Pos          Squad           Comp  \
5        6  Yunis Abdelhamid  ma MAR     DF  Saint-Étienne     fr Ligue 1   
8        9          Ali Abdi  tn TUN  DF,MF           Nice     fr Ligue 1   
10      11              Abel  es ESP     DF        Osasuna     es La Liga   
14      15             Abner  br BRA     DF           Lyon     fr Ligue 1   
16      17       Abdel Abqar  ma MAR     DF         Alavés     es La Liga   
...    ...               ...     ...    ...            ...            ...   
2833  2834    Cedric Zesiger  ch SUI     DF       Augsburg  de Bundesliga   
2836  2837       Nathan Zeze  fr FRA     DF         Nantes     fr Ligue 1   
2844  2845      Nadir Zortea  it ITA  DF,MF       Cagliari     it Serie A   
2846  2847     Yanis Zouaoui  dz ALG  DF,FW       Le Havre     fr Ligue 1   
2847  2848     Igor Zubeldia  es ESP     DF  Real Sociedad     es La Liga   

       Age    Born  MP  Starts  ...  Att (GK)  Thr  Launch%  AvgLen  Opp  S

In [15]:
meta_cols = ['Player','Squad','Comp','Nation','Pos','Age','Born','Foot']
numeric_cols = [col for col in df_cbs.columns if col not in meta_cols]

In [16]:
for col in numeric_cols:
    df_cbs[col]=pd.to_numeric(df_cbs[col],errors='coerce')

df_cbs[numeric_cols]=df_cbs[numeric_cols].fillna(df_cbs[numeric_cols].median())

In [18]:
martinez_rows=df_cbs[df_cbs['Player'].str.contains('Martínez',na=False)]
martinez_idx = martinez_rows.index[0]
print(df_cbs.loc[martinez_idx])

Rk                   1648
Player     Iñigo Martínez
Nation             es ESP
Pos                    DF
Squad           Barcelona
                ...      
Stp                   NaN
Stp%                  NaN
#OPA                  NaN
#OPA/90               NaN
AvgDist               NaN
Name: 1647, Length: 267, dtype: object


In [19]:
outlier_analysis=[]

for col in numeric_cols:
    mean_val = df_cbs[col].mean()
    std_val = df_cbs[col].std()

    # avoid division by 0
    if std_val>0:
        z_score=(df_cbs.loc[martinez_idx,col]-mean_val)/std_val
        percentile=df_cbs[col].rank(pct=True).loc[martinez_idx]*100

        outlier_analysis.append({
            'Metric':col,
            'Martinez_value':df_cbs.loc[martinez_idx,col],
            'Average':round(mean_val,2),
            'Z_Score':round(z_score,2),
            'Percentile':round(percentile,1)
        })
        

In [23]:
df_outliers = pd.DataFrame(outlier_analysis)

top_anomalies = df_outliers.sort_values(by='Z_Score',ascending=False).head(20)
print("inigo martinez's most unique attributes: ")
print(top_anomalies.to_string(index=False))

inigo martinez's most unique attributes: 
                  Metric  Martinez_value  Average  Z_Score  Percentile
PrgDist_stats_possession          7336.0  2001.14     4.75       100.0
                      Sw            34.0     5.87     4.50        99.3
                    PrgP           260.0    79.70     4.15        99.7
      PrgP_stats_passing           260.0    79.70     4.15        99.7
                     1/3           259.0    72.04     4.06        99.7
TotDist_stats_possession         10196.0  3764.42     3.44        99.6
Mid 3rd_stats_possession          1629.0   652.53     3.22        99.4
  FK_stats_passing_types            94.0    29.33     3.15        99.2
                    onxG            70.2    29.56     3.08        99.9
                     onG            75.0    30.89     2.95        99.6
                 TotDist         41890.0 17284.67     2.90        99.0
                 PrgDist         14576.0  5986.85     2.79        98.4
                   xG+/-           

In [43]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# Use the top anomaly metrics as the KNN feature set
knn_features = [
    metric for metric in top_anomalies['Metric']
    if metric in df_cbs.columns
]


# Build a CBS feature matrix for only those metrics
X = df_cbs[knn_features].apply(pd.to_numeric, errors='coerce')
X = X.fillna(X.median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

knn = NearestNeighbors(n_neighbors=100, metric='euclidean')
knn.fit(X_scaled)

# Find Martinez row among CBS players
martinez_idx = df_cbs[df_cbs['Player'].str.contains('Martínez', na=False)].index[0]
martinez_pos = df_cbs.index.get_loc(martinez_idx)

distances, indices = knn.kneighbors(X_scaled[martinez_pos:martinez_pos + 1])
neighbors = df_cbs.iloc[indices[0]].copy()
neighbors['Distance'] = distances[0]

neighbors = neighbors[neighbors['Age']<30]

print("Top CBS nearest neighbors using top anomaly features:")
print(neighbors[['Player', 'Squad', 'Pos', 'Age']+ ['Distance'] + knn_features].to_string(index=False) )

Top CBS nearest neighbors using top anomaly features:
                Player           Squad   Pos  Age  Distance  PrgDist_stats_possession  Sw  PrgP  PrgP_stats_passing  1/3  TotDist_stats_possession  Mid 3rd_stats_possession  FK_stats_passing_types  onxG  onG  TotDist  PrgDist  xG+/-  Carries  Rec  Cmp  Cmp_stats_passing_types  Live  +/-  PKwon
    Jan Paul van Hecke        Brighton    DF 24.0  5.662424                      7191  19   228                 228  226                     10633                      1350                     104  49.7   55    39486    15217    5.4     1715 1820 2016                     2016  2146    8      0
        Joško Gvardiol Manchester City    DF 22.0  5.951743                      6002  11   264                 264  247                     10574                      1549                      41  66.4   70    39905    10760   23.1     1997 2257 2454                     2454  2502   29      0
     Federico Valverde     Real Madrid MF,DF 26.0  6.156634  